# VQE Optimization

This tutorial demonstrates how to use qiskit-trev's built-in optimizers to find the ground state energy of a Hamiltonian using the Variational Quantum Eigensolver (VQE) approach.

We cover two optimization strategies:
1. **Gradient descent** — uses parameter-shift gradients with Adam/SGD
2. **CMA-ES** — derivative-free, population-based evolutionary strategy

In [ ]:
import math
import torch
import matplotlib.pyplot as plt
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_trev import TensorRingModel, GradientOptimizer, CMAESOptimizer

## 1. Problem Setup

We'll minimize a simple 2-qubit Hamiltonian using a parameterized ansatz.

**Hamiltonian:** H = ZI + IZ (sum of single-qubit Z operators)

**Ansatz:** RY on each qubit. The ground state is |11> with energy -2.0.

In [ ]:
# Ansatz: RY(t0)|0> x RY(t1)|0>
qc = QuantumCircuit(2)
qc.ry(0.0, 0)
qc.ry(0.0, 1)

# Hamiltonian: ZI + IZ
H = SparsePauliOp.from_list([("ZI", 1.0), ("IZ", 1.0)])

model = TensorRingModel(qc, H, rank=4, device="cpu")

# Initial parameters (random starting point)
theta0 = torch.tensor([0.5, 0.3])
print(f"Initial energy: {model(theta0).item():.4f}")
print(f"Target (ground state): -2.0")

## 2. Gradient Descent Optimizer

`GradientOptimizer` uses batched parameter-shift gradients with a PyTorch optimizer (Adam or SGD) under the hood.

In [ ]:
grad_opt = GradientOptimizer(lr=0.1, optimizer_cls="adam")
result_grad = grad_opt.minimize(model, theta0.clone(), max_iter=50)

print(f"Final energy:  {result_grad.cost:.4f}")
print(f"Final params:  {result_grad.params.tolist()}")
print(f"Optimal theta: [pi, pi] = [{math.pi:.4f}, {math.pi:.4f}]")

## 3. CMA-ES Optimizer

`CMAESOptimizer` is a derivative-free optimizer that maintains a population of candidate solutions. It's useful when the cost landscape is noisy or has many local minima.

In [ ]:
cma_opt = CMAESOptimizer(sigma=0.5, pop_size=10)
result_cma = cma_opt.minimize(model, theta0.clone(), max_iter=50)

print(f"Final energy:  {result_cma.cost:.4f}")
print(f"Final params:  {result_cma.params.tolist()}")

## 4. Comparing Convergence

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(result_grad.cost_history, label="Gradient (Adam)")
plt.plot(result_cma.cost_history, label="CMA-ES")
plt.axhline(y=-2.0, color="k", linestyle="--", label="Ground state")
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.legend()
plt.title("VQE Convergence: Gradient vs CMA-ES")
plt.tight_layout()
plt.show()

## 5. A Larger Example: 4-Qubit Ising Model

Let's try a more realistic problem — a 4-qubit transverse-field Ising model with a deeper ansatz.

In [ ]:
# 4-qubit ansatz: layer of RY + entangling CNOTs, repeated twice
n_qubits = 4
n_layers = 2

qc = QuantumCircuit(n_qubits)
for layer in range(n_layers):
    for q in range(n_qubits):
        qc.ry(0.0, q)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)

n_params = n_qubits * n_layers
print(f"Ansatz: {n_qubits} qubits, {n_layers} layers, {n_params} parameters")

# Transverse-field Ising: H = sum ZZ + 0.5 * sum X
H_ising = SparsePauliOp.from_list([
    ("ZZII", 1.0), ("IZZI", 1.0), ("IIZZ", 1.0),  # ZZ nearest-neighbor
    ("XIII", -0.5), ("IXII", -0.5), ("IIXI", -0.5), ("IIIX", -0.5),  # transverse field
])

model = TensorRingModel(qc, H_ising, rank=8, device="cpu")

In [ ]:
# Optimize with gradient descent
theta0 = torch.randn(n_params) * 0.1

grad_opt = GradientOptimizer(lr=0.05, optimizer_cls="adam")
result = grad_opt.minimize(model, theta0, max_iter=100)

plt.figure(figsize=(8, 4))
plt.plot(result.cost_history)
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title(f"4-Qubit Ising VQE (final energy: {result.cost:.4f})")
plt.tight_layout()
plt.show()

print(f"Optimized energy: {result.cost:.4f}")

## 6. Manual Optimization Loop

You can also build your own optimization loop using `parameter_shift_grad` directly. This gives you full control over the training process.

In [ ]:
# Simple gradient descent on the 2-qubit problem
qc = QuantumCircuit(2)
qc.ry(0.0, 0)
qc.ry(0.0, 1)
H = SparsePauliOp.from_list([("ZI", 1.0), ("IZ", 1.0)])
model = TensorRingModel(qc, H, rank=4, device="cpu")

theta = torch.tensor([0.5, 0.3])
lr = 0.3
history = []

for step in range(30):
    energy = model(theta).item()
    history.append(energy)
    grad = model.parameter_shift_grad(theta)
    theta = theta - lr * grad

print(f"Final energy: {history[-1]:.4f}  (target: -2.0)")
print(f"Final theta:  {theta.tolist()}")

plt.figure(figsize=(8, 4))
plt.plot(history, "o-", markersize=3)
plt.axhline(y=-2.0, color="k", linestyle="--", label="Ground state")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.legend()
plt.title("Manual gradient descent")
plt.tight_layout()
plt.show()